# SKAB Anomaly Detection — Feature Engineering + Ensemble Meta-Learner

**Approach:** Rich temporal feature engineering + IsolationForest / OneClassSVM / LOF / Dense-AE ensemble  
**Meta-learner:** LightGBM trained on temporal validation fold  
**Target:** ROC-AUC > 0.80

In [ ]:
# ── GPU compatibility shim (P100 / T4 / A100 support) ──────────────
import subprocess, sys

def _gpu_name():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']
        ).decode()
        return out.strip().splitlines()[0] if out.strip() else ''
    except Exception:
        return ''

_GPU = _gpu_name()
print(f'Detected GPU: {_GPU!r}')

if 'P100' in _GPU:
    print('Tesla P100 — installing torch 2.3.1+cu118 for sm_60 support...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.3.1', 'torchvision==0.18.1',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    ])
    for _m in list(sys.modules.keys()):
        if _m == 'torch' or _m.startswith('torch.'):
            del sys.modules[_m]

# Install remaining packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'lightgbm', 'scikit-learn'])

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    roc_curve, confusion_matrix
)

import lightgbm as lgb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | LightGBM: {lgb.__version__} | Device: {DEVICE}')

START_TIME = time.time()

In [ ]:
# ──────────────────────────────────────────────────────────
# 1. Data Loading
# ──────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/skoltech-anomaly-benchmark-skab')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
    'Pressure', 'Temperature', 'Thermocouple',
    'Voltage', 'Volume Flow RateRMS'
]
LABEL_COL = 'anomaly'


def load_skab(base_path: Path) -> pd.DataFrame:
    """Load all SKAB csv files, tag with file source, sort by datetime."""
    dfs = []
    for csv_path in sorted(base_path.rglob('*.csv')):
        try:
            df = pd.read_csv(csv_path, sep=';', parse_dates=['datetime'],
                             index_col='datetime')
            df.columns = [c.strip() for c in df.columns]
            required = set(FEATURE_COLS + [LABEL_COL])
            if not required.issubset(df.columns):
                continue
            df = df[FEATURE_COLS + [LABEL_COL]].copy()
            df['file_id'] = str(csv_path.relative_to(base_path))
            df = df.dropna(subset=FEATURE_COLS)
            dfs.append(df)
        except Exception as e:
            print(f'  Skip {csv_path.name}: {e}')
    combined = pd.concat(dfs, axis=0).sort_index()
    combined[LABEL_COL] = combined[LABEL_COL].astype(int)
    return combined


print('Loading SKAB data...')
data = load_skab(BASE)
print(f'Total rows     : {len(data):,}')
print(f'Anomaly rate   : {data[LABEL_COL].mean():.3%}')
print(f'Files          : {data["file_id"].nunique()}')
print(data[FEATURE_COLS].describe().T[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
# ──────────────────────────────────────────────────────────
# 2. Feature Engineering
# ──────────────────────────────────────────────────────────
WINDOWS   = [5, 10, 30, 60]
TOP_K_FFT = 5
FFT_WIN   = 64


def make_features(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Build rich feature matrix (rolling stats, FFT, CUSUM, cross-corr)."""
    feats = {}

    # Raw values
    for c in cols:
        feats[c] = df[c].values

    for c in cols:
        x = df[c]

        # Rate of change
        feats[f'{c}_diff1'] = x.diff(1).fillna(0).values
        feats[f'{c}_diff2'] = x.diff(2).fillna(0).values
        pct = x.pct_change(fill_method=None).replace([np.inf, -np.inf], 0).fillna(0)
        feats[f'{c}_pct'] = pct.values

        # Rolling statistics
        for w in WINDOWS:
            roll = x.rolling(w, min_periods=1)
            feats[f'{c}_rmean_{w}']  = roll.mean().values
            feats[f'{c}_rstd_{w}']   = roll.std().fillna(0).values
            feats[f'{c}_rmin_{w}']   = roll.min().values
            feats[f'{c}_rmax_{w}']   = roll.max().values
            feats[f'{c}_rrange_{w}'] = (roll.max() - roll.min()).fillna(0).values
            # Rolling Z-score
            mu    = roll.mean()
            sigma = roll.std().fillna(1e-8).replace(0, 1e-8)
            feats[f'{c}_rzs_{w}'] = ((x - mu) / sigma).fillna(0).values
            if w >= 30:
                feats[f'{c}_rskew_{w}'] = roll.skew().fillna(0).values

    # CUSUM
    for c in cols:
        mu_c = df[c].mean()
        arr  = df[c].values
        feats[f'{c}_cusum_pos'] = np.maximum.accumulate(np.cumsum(arr - mu_c - 0.5))
        feats[f'{c}_cusum_neg'] = np.minimum.accumulate(np.cumsum(arr - mu_c + 0.5))

    # Cross-feature rolling correlations
    pairs = [
        ('Accelerometer1RMS', 'Accelerometer2RMS'),
        ('Current', 'Voltage'),
        ('Pressure', 'Volume Flow RateRMS'),
        ('Temperature', 'Thermocouple'),
    ]
    for c1, c2 in pairs:
        if c1 in df.columns and c2 in df.columns:
            for w in [30, 60]:
                cor = df[c1].rolling(w, min_periods=5).corr(df[c2]).fillna(0)
                feats[f'corr_{c1[:4]}_{c2[:4]}_{w}'] = cor.values

    # FFT magnitudes over rolling window
    for c in cols:
        arr = df[c].values.astype(np.float32)
        fft_feats = np.zeros((len(arr), TOP_K_FFT), dtype=np.float32)
        for i in range(len(arr)):
            start   = max(0, i - FFT_WIN + 1)
            seg     = arr[start:i + 1]
            if len(seg) < 4:
                continue
            mags = np.abs(np.fft.rfft(seg - seg.mean()))
            k    = min(TOP_K_FFT, len(mags))
            fft_feats[i, :k] = np.sort(mags)[::-1][:k]
        for k in range(TOP_K_FFT):
            feats[f'{c}_fft_{k}'] = fft_feats[:, k]

    feat_df = pd.DataFrame(feats, index=df.index)
    feat_df.replace([np.inf, -np.inf], 0, inplace=True)
    feat_df.fillna(0, inplace=True)
    return feat_df


print('Engineering features per file...')
feat_chunks, label_chunks = [], []

for file_id, grp in data.groupby('file_id', sort=False):
    grp_sorted = grp.sort_index()
    f = make_features(grp_sorted, FEATURE_COLS)
    feat_chunks.append(f)
    label_chunks.append(grp_sorted[LABEL_COL])
    print(f'  {file_id}: {len(grp_sorted)} rows, {f.shape[1]} features')

X_all = pd.concat(feat_chunks, axis=0)
y_all = pd.concat(label_chunks, axis=0)
print(f'\nFull feature matrix: {X_all.shape}')
print(f'NaN count           : {X_all.isna().sum().sum()}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 3. Temporal split  70% train | 15% val | 15% test
# ──────────────────────────────────────────────────────────
n = len(X_all)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train_raw = X_all.iloc[:train_end]
y_train     = y_all.iloc[:train_end]

X_val_raw   = X_all.iloc[train_end:val_end]
y_val       = y_all.iloc[train_end:val_end]

X_test_raw  = X_all.iloc[val_end:]
y_test      = y_all.iloc[val_end:]

print(f'Train : {len(X_train_raw):,}  anomaly={y_train.mean():.3%}')
print(f'Val   : {len(X_val_raw):,}  anomaly={y_val.mean():.3%}')
print(f'Test  : {len(X_test_raw):,}  anomaly={y_test.mean():.3%}')

scaler  = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)
print('Scaling done.')

CONTAMINATION = float(np.clip(y_train.mean(), 0.01, 0.45))
print(f'Contamination rate: {CONTAMINATION:.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 4. Isolation Forest
# ──────────────────────────────────────────────────────────
print('Training Isolation Forest...')
iso = IsolationForest(
    n_estimators=400,
    contamination=CONTAMINATION,
    max_features=0.8,
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_train)

iso_train = -iso.score_samples(X_train)
iso_val   = -iso.score_samples(X_val)
iso_test  = -iso.score_samples(X_test)

print(f'  Val  ROC-AUC: {roc_auc_score(y_val, iso_val):.4f}')
print(f'  Test ROC-AUC: {roc_auc_score(y_test, iso_test):.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 5. One-Class SVM  (subsampled for speed)
# ──────────────────────────────────────────────────────────
print('Training One-Class SVM...')
MAX_SVM = 20000
rng = np.random.RandomState(42)
idx_svm = rng.choice(len(X_train), min(MAX_SVM, len(X_train)), replace=False)

ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=CONTAMINATION, cache_size=2000)
ocsvm.fit(X_train[idx_svm])

ocsvm_train = -ocsvm.score_samples(X_train)
ocsvm_val   = -ocsvm.score_samples(X_val)
ocsvm_test  = -ocsvm.score_samples(X_test)

print(f'  Val  ROC-AUC: {roc_auc_score(y_val, ocsvm_val):.4f}')
print(f'  Test ROC-AUC: {roc_auc_score(y_test, ocsvm_test):.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 6. Local Outlier Factor  (novelty mode)
# ──────────────────────────────────────────────────────────
print('Training LOF...')
MAX_LOF = 30000
idx_lof = rng.choice(len(X_train), min(MAX_LOF, len(X_train)), replace=False)

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    novelty=True,
    n_jobs=-1,
)
lof.fit(X_train[idx_lof])

lof_train = -lof.score_samples(X_train)
lof_val   = -lof.score_samples(X_val)
lof_test  = -lof.score_samples(X_test)

print(f'  Val  ROC-AUC: {roc_auc_score(y_val, lof_val):.4f}')
print(f'  Test ROC-AUC: {roc_auc_score(y_test, lof_test):.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 7. Dense Autoencoder  (reconstruction error)
# ──────────────────────────────────────────────────────────
class DenseAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        hidden = [256, 128, 64]
        # Encoder
        enc, d = [], input_dim
        for h in hidden:
            enc += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.1)]
            d = h
        enc += [nn.Linear(d, latent_dim)]
        self.encoder = nn.Sequential(*enc)
        # Decoder
        dec, d = [], latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.1)]
            d = h
        dec += [nn.Linear(d, input_dim)]
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_ae(X_tr, X_va, epochs=80, batch_size=512, lr=1e-3, patience=12):
    D      = X_tr.shape[1]
    model  = DenseAE(D, latent_dim=32).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit   = nn.MSELoss()
    tr_t   = torch.tensor(X_tr, dtype=torch.float32)
    va_t   = torch.tensor(X_va, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tr_t), batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=True)

    best_val, best_state, no_imp, history = float('inf'), None, 0, []

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            out  = model(xb)
            loss = crit(out, xb)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item() * len(xb)
        ep_loss /= len(tr_t)
        sched.step()

        model.eval()
        with torch.no_grad():
            va_out = model(va_t.to(DEVICE))
            val_loss = crit(va_out, va_t.to(DEVICE)).item()
        history.append((ep_loss, val_loss))

        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'    Early stop @ epoch {epoch}')
                break

        if epoch % 20 == 0:
            print(f'    Epoch {epoch:3d} | train {ep_loss:.5f} | val {val_loss:.5f}')

    model.load_state_dict(best_state)
    return model, history


def ae_errors(model, X):
    model.eval()
    t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        out = model(t)
    return ((t - out) ** 2).mean(dim=1).cpu().numpy()


print('Training Dense Autoencoder...')
ae_model, ae_history = train_ae(X_train, X_val, epochs=80, patience=12)

ae_train = ae_errors(ae_model, X_train)
ae_val   = ae_errors(ae_model, X_val)
ae_test  = ae_errors(ae_model, X_test)

print(f'  Val  ROC-AUC: {roc_auc_score(y_val, ae_val):.4f}')
print(f'  Test ROC-AUC: {roc_auc_score(y_test, ae_test):.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 8. Rank-normalise base-learner scores to [0,1]
# ──────────────────────────────────────────────────────────
def rank_norm(train_s, *others):
    """Percentile clip + min-max using training distribution."""
    lo  = np.percentile(train_s, 1)
    hi  = np.percentile(train_s, 99)
    rng = hi - lo if hi != lo else 1.0
    def _n(s): return np.clip((s - lo) / rng, 0, 1)
    return (_n(train_s),) + tuple(_n(s) for s in others)


iso_tr_n,   iso_va_n,   iso_te_n   = rank_norm(iso_train,   iso_val,   iso_test)
ocsvm_tr_n, ocsvm_va_n, ocsvm_te_n = rank_norm(ocsvm_train, ocsvm_val, ocsvm_test)
lof_tr_n,   lof_va_n,   lof_te_n   = rank_norm(lof_train,   lof_val,   lof_test)
ae_tr_n,    ae_va_n,    ae_te_n    = rank_norm(ae_train,    ae_val,    ae_test)

print('Scores normalised.')

In [ ]:
# ──────────────────────────────────────────────────────────
# 9. Build meta-feature matrix
# ──────────────────────────────────────────────────────────
def build_meta(scores_list, orig_X_scaled):
    base  = np.stack(scores_list, axis=1)                     # (N, 4)
    avg   = base.mean(axis=1, keepdims=True)
    mx    = base.max(axis=1, keepdims=True)
    top2  = np.sort(base, axis=1)[:, -2:].prod(axis=1, keepdims=True)
    var   = base.var(axis=1, keepdims=True)
    extra = np.concatenate([avg, mx, top2, var], axis=1)      # (N, 4)
    return np.concatenate([base, extra, orig_X_scaled], axis=1)


meta_X_train = build_meta([iso_tr_n,  ocsvm_tr_n,  lof_tr_n,  ae_tr_n],  X_train)
meta_X_val   = build_meta([iso_va_n,  ocsvm_va_n,  lof_va_n,  ae_va_n],  X_val)
meta_X_test  = build_meta([iso_te_n,  ocsvm_te_n,  lof_te_n,  ae_te_n],  X_test)

print(f'Meta shapes: train={meta_X_train.shape}, val={meta_X_val.shape}, test={meta_X_test.shape}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 10. LightGBM Meta-Learner  (train on val, evaluate on test)
# ──────────────────────────────────────────────────────────
print('Training LightGBM meta-learner...')

pos_w = float((y_val == 0).sum()) / float((y_val == 1).sum() + 1e-8)

lgb_tr_ds  = lgb.Dataset(meta_X_val,  label=y_val.values)
lgb_val_ds = lgb.Dataset(meta_X_test, label=y_test.values, reference=lgb_tr_ds)

params = {
    'objective':       'binary',
    'metric':          'auc',
    'boosting_type':   'gbdt',
    'num_leaves':      63,
    'learning_rate':   0.03,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':    5,
    'min_child_samples': 20,
    'lambda_l1':       0.1,
    'lambda_l2':       0.1,
    'verbose':        -1,
    'n_jobs':         -1,
    'seed':            42,
    'scale_pos_weight': pos_w,
}

meta_model = lgb.train(
    params,
    lgb_tr_ds,
    num_boost_round=1000,
    valid_sets=[lgb_val_ds],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)

meta_pred_val  = meta_model.predict(meta_X_val)
meta_pred_test = meta_model.predict(meta_X_test)

print(f'  Val  ROC-AUC: {roc_auc_score(y_val, meta_pred_val):.4f}')
print(f'  Test ROC-AUC: {roc_auc_score(y_test, meta_pred_test):.4f}')
print(f'  Best iteration: {meta_model.best_iteration}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 11. Compare all approaches on TEST set
# ──────────────────────────────────────────────────────────
def best_f1_thresh(y_true, scores):
    thresholds = np.percentile(scores, np.arange(50, 99, 0.5))
    best_f1, best_t = 0.0, thresholds[0]
    for t in thresholds:
        preds = (scores >= t).astype(int)
        f = f1_score(y_true, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_f1, best_t


simple_avg = (iso_te_n + ocsvm_te_n + lof_te_n + ae_te_n) / 4

results = {}
for name, scores in [
    ('IsolationForest', iso_te_n),
    ('OneClassSVM',     ocsvm_te_n),
    ('LOF',             lof_te_n),
    ('DenseAE',         ae_te_n),
    ('SimpleAverage',   simple_avg),
    ('LightGBM_meta',   meta_pred_test),
]:
    auc      = roc_auc_score(y_test, scores)
    pr       = average_precision_score(y_test, scores)
    f1, t    = best_f1_thresh(y_test, scores)
    results[name] = dict(roc_auc=auc, pr_auc=pr, best_f1=f1, threshold=t)
    print(f'{name:20s}  ROC-AUC={auc:.4f}  PR-AUC={pr:.4f}  F1={f1:.4f}')

best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f'\nBest: {best_name}  ROC-AUC={results[best_name]["roc_auc"]:.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 12. Per-file evaluation with LightGBM ensemble
# ──────────────────────────────────────────────────────────
print('Per-file evaluation...')

all_X_scaled = np.concatenate([X_train, X_val, X_test], axis=0)
all_iso   = np.concatenate([iso_tr_n,   iso_va_n,   iso_te_n])
all_ocsvm = np.concatenate([ocsvm_tr_n, ocsvm_va_n, ocsvm_te_n])
all_lof   = np.concatenate([lof_tr_n,   lof_va_n,   lof_te_n])
all_ae    = np.concatenate([ae_tr_n,    ae_va_n,    ae_te_n])

meta_X_all   = build_meta([all_iso, all_ocsvm, all_lof, all_ae], all_X_scaled)
all_ensemble = meta_model.predict(meta_X_all)

file_ids_arr = data['file_id'].values
y_all_arr    = y_all.values
per_file     = {}

for fid in np.unique(file_ids_arr):
    mask = file_ids_arr == fid
    ys   = y_all_arr[mask]
    sc   = all_ensemble[mask]
    if ys.sum() < 1 or ys.sum() == len(ys):
        continue
    try:
        auc = roc_auc_score(ys, sc)
        pr  = average_precision_score(ys, sc)
        per_file[fid] = dict(roc_auc=auc, pr_auc=pr,
                             n_samples=int(mask.sum()),
                             anomaly_rate=float(ys.mean()))
        print(f'  {fid:40s} AUC={auc:.4f}  N={mask.sum()}')
    except Exception as e:
        print(f'  Skip {fid}: {e}')

mean_per_file_auc = float(np.mean([v['roc_auc'] for v in per_file.values()]))
print(f'\nMean per-file ROC-AUC: {mean_per_file_auc:.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 13. Save metrics.json
# ──────────────────────────────────────────────────────────
best_res    = results[best_name]
best_scores = meta_pred_test if best_name == 'LightGBM_meta' else simple_avg
best_preds  = (best_scores >= best_res['threshold']).astype(int)
cm          = confusion_matrix(y_test, best_preds).tolist()

metrics = {
    'roc_auc':           best_res['roc_auc'],
    'pr_auc':            best_res['pr_auc'],
    'best_f1':           best_res['best_f1'],
    'f1_macro':          float(f1_score(y_test, best_preds, average='macro', zero_division=0)),
    'threshold':         float(best_res['threshold']),
    'train_time_s':      time.time() - START_TIME,
    'best_model':        best_name,
    'n_features':        int(X_all.shape[1]),
    'feature_columns':   FEATURE_COLS,
    'confusion_matrix':  cm,
    'all_models':        results,
    'per_file_auc_mean': mean_per_file_auc,
    'config': {
        'windows':             WINDOWS,
        'top_k_fft':           TOP_K_FFT,
        'ae_latent_dim':       32,
        'lgb_best_iteration':  int(meta_model.best_iteration),
        'contamination_rate':  float(CONTAMINATION),
        'train_size':          len(X_train),
        'val_size':            len(X_val),
        'test_size':           len(X_test),
    },
}

metrics_path = OUTPUT_DIR / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved: {metrics_path}')
print(f'ROC-AUC  : {metrics["roc_auc"]:.4f}')
print(f'PR-AUC   : {metrics["pr_auc"]:.4f}')
print(f'Best F1  : {metrics["best_f1"]:.4f}')
print(f'F1-macro : {metrics["f1_macro"]:.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 14. training_results.png
# ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

MODEL_COLORS = {
    'IsolationForest': '#E74C3C',
    'OneClassSVM':     '#3498DB',
    'LOF':             '#2ECC71',
    'DenseAE':         '#F39C12',
    'SimpleAverage':   '#9B59B6',
    'LightGBM_meta':   '#1ABC9C',
}

TEST_SCORES = {
    'IsolationForest': iso_te_n,
    'OneClassSVM':     ocsvm_te_n,
    'LOF':             lof_te_n,
    'DenseAE':         ae_te_n,
    'SimpleAverage':   simple_avg,
    'LightGBM_meta':   meta_pred_test,
}

# ── ROC curves
ax1 = fig.add_subplot(gs[0, 0])
for name, scores in TEST_SCORES.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = results[name]['roc_auc']
    lw  = 2.5 if name == best_name else 1.2
    ax1.plot(fpr, tpr, color=MODEL_COLORS[name], lw=lw,
             label=f'{name} ({auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curves — Test Set')
ax1.legend(fontsize=6.5); ax1.grid(alpha=0.3)

# ── ROC-AUC bar chart
ax2 = fig.add_subplot(gs[0, 1])
names = list(results.keys())
aucs  = [results[n]['roc_auc'] for n in names]
bars  = ax2.bar(range(len(names)), aucs, color=[MODEL_COLORS[n] for n in names])
ax2.axhline(0.80, color='red', linestyle='--', lw=1.5, label='Target 0.80')
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(names, rotation=45, ha='right', fontsize=7.5)
ax2.set_title('ROC-AUC per Model')
ax2.set_ylim(0.4, 1.0)
for bar, auc in zip(bars, aucs):
    ax2.text(bar.get_x() + bar.get_width() / 2, auc + 0.005,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=7)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# ── AE training loss
ax3 = fig.add_subplot(gs[0, 2])
if ae_history:
    ax3.plot([h[0] for h in ae_history], label='Train', color='#E74C3C')
    ax3.plot([h[1] for h in ae_history], label='Val',   color='#3498DB')
ax3.set_title('Dense AE Training Loss')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('MSE')
ax3.legend(); ax3.grid(alpha=0.3)

# ── Score distributions
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(meta_pred_test[y_test == 0], bins=60, alpha=0.6,
         color='steelblue', label='Normal', density=True)
ax4.hist(meta_pred_test[y_test == 1], bins=60, alpha=0.6,
         color='crimson',   label='Anomaly', density=True)
ax4.axvline(best_res['threshold'], color='orange', lw=2,
            label=f'Thr={best_res["threshold"]:.3f}')
ax4.set_title('LightGBM Score Distributions (Test)')
ax4.set_xlabel('Anomaly Score'); ax4.set_ylabel('Density')
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# ── Per-file ROC-AUC
ax5 = fig.add_subplot(gs[1, 1:3])
fids      = list(per_file.keys())
f_aucs    = [per_file[f]['roc_auc'] for f in fids]
si        = np.argsort(f_aucs)[::-1]
s_fids    = [fids[i].split('/')[-1].replace('.csv', '') for i in si]
s_aucs    = [f_aucs[i] for i in si]
c_pf      = ['#2ECC71' if a >= 0.80 else '#E74C3C' for a in s_aucs]
ax5.barh(range(len(s_fids)), s_aucs, color=c_pf)
ax5.axvline(0.80, color='darkgreen', linestyle='--', lw=1.5, label='Target 0.80')
ax5.axvline(mean_per_file_auc, color='navy', linestyle='-', lw=1.5,
            label=f'Mean={mean_per_file_auc:.3f}')
ax5.set_yticks(range(len(s_fids)))
ax5.set_yticklabels(s_fids, fontsize=7)
ax5.set_title('Per-File ROC-AUC (LightGBM Ensemble)')
ax5.set_xlim(0, 1.05); ax5.legend(fontsize=8); ax5.grid(axis='x', alpha=0.3)

# ── Confusion matrix
ax6 = fig.add_subplot(gs[2, 0])
im  = ax6.imshow(np.array(cm), interpolation='nearest', cmap='Blues')
ax6.set_title(f'Confusion Matrix ({best_name})')
ax6.set_xlabel('Predicted'); ax6.set_ylabel('True')
ax6.set_xticks([0, 1]); ax6.set_yticks([0, 1])
ax6.set_xticklabels(['Normal', 'Anomaly'])
ax6.set_yticklabels(['Normal', 'Anomaly'])
max_val = np.array(cm).max()
for i in range(2):
    for j in range(2):
        ax6.text(j, i, str(cm[i][j]), ha='center', va='center',
                 color='white' if cm[i][j] > max_val / 2 else 'black', fontsize=14)

# ── LightGBM feature importances
ax7 = fig.add_subplot(gs[2, 1:3])
fi      = meta_model.feature_importance(importance_type='gain')
top_n   = 30
top_idx = np.argsort(fi)[::-1][:top_n]
meta_feat_names = (
    ['IF_score', 'SVM_score', 'LOF_score', 'AE_score',
     'avg_score', 'max_score', 'top2_prod', 'var_score'] +
    list(X_all.columns)
)
labels_top = [
    meta_feat_names[i] if i < len(meta_feat_names) else f'feat_{i}'
    for i in top_idx
]
ax7.barh(range(top_n), fi[top_idx], color='steelblue')
ax7.set_yticks(range(top_n))
ax7.set_yticklabels(labels_top, fontsize=6.5)
ax7.set_title('Top-30 LightGBM Feature Importances (gain)')
ax7.set_xlabel('Importance (gain)'); ax7.grid(axis='x', alpha=0.3)

fig.suptitle(
    f'SKAB Anomaly Detection — Feature Engineering + Ensemble\n'
    f'Best: {best_name}  '
    f'ROC-AUC={best_res["roc_auc"]:.4f}  '
    f'PR-AUC={best_res["pr_auc"]:.4f}  '
    f'F1={best_res["best_f1"]:.4f}',
    fontsize=13, fontweight='bold',
)

plot_path = OUTPUT_DIR / 'training_results.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Plot saved: {plot_path}')

In [ ]:
# ──────────────────────────────────────────────────────────
# 15. Final summary
# ──────────────────────────────────────────────────────────
elapsed = time.time() - START_TIME
print('=' * 68)
print('FINAL RESULTS — SKAB Feature Engineering + Ensemble')
print('=' * 68)
for name in results:
    r   = results[name]
    tag = ' <-- BEST' if name == best_name else ''
    print(f'{name:20s}  AUC={r["roc_auc"]:.4f}  PR={r["pr_auc"]:.4f}  F1={r["best_f1"]:.4f}{tag}')
print('-' * 68)
print(f'Mean per-file AUC  : {mean_per_file_auc:.4f}')
print(f'Feature matrix dim : {X_all.shape[1]}')
print(f'Total time         : {elapsed / 60:.1f} min')
print(f'Target (>0.80) MET : {best_res["roc_auc"] >= 0.80}')
print('=' * 68)